# Neuron Connectivity Analysis

Analyses the recurrent weight matrix **W_rec** of a trained LeakyRNN (shape `n_rnn × n_rnn`,
where `W_rec[i, j]` is the weight **from** neuron i **to** neuron j via `h @ W_rec`).

| Section | What it shows |
|---------|---------------|
| 1. E/I classification | Net outgoing weight per neuron (row sum of W_rec) |
| 2. Hierarchical clustering | Neurons grouped by outgoing weight profile (cosine distance) |
| 3. Sorted heatmap | W_rec re-ordered by cluster — reveals block structure |
| 4. Within vs between | Mean \|weight\| inside vs across clusters (modularity) |
| 5. Functional profiles | Input selectivity (W_in) and readout contribution (W_out) per cluster |
| 6. PCA embedding | Neurons in connectivity PC space, coloured by cluster / E-I |


In [ ]:
import os, sys, json, pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist
from sklearn.decomposition import PCA

%matplotlib inline
plt.rcParams.update({'font.size': 11})

In [ ]:
# ── configuration ─────────────────────────────────────────────────────────────
env_file = Path.cwd().parent / '.env'
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if '=' in line and not line.startswith('#'):
            key, _, val = line.partition('=')
            os.environ.setdefault(key.strip(), val.strip())

results_dir = Path(os.environ.get('TONE_RESULTS_DIR', ''))
if not results_dir:
    raise RuntimeError('TONE_RESULTS_DIR is not set. Add it to .env or export it.')

CKPT       = results_dir / 'harmonic/harmonic_0_lr0.0001_bs64_noise0.1_nrnn128/checkpoint_10000.pkl'
N_CLUSTERS = 4     # number of connectivity clusters (adjust after looking at the dendrogram)
LINK_METHOD = 'ward'  # linkage method: 'ward', 'average', 'complete'

In [ ]:
# ── load weights ──────────────────────────────────────────────────────────────
with open(CKPT.parent / 'config.json') as f:
    config = json.load(f)

hp          = config['training']
task_params = config['task_params']

with open(CKPT, 'rb') as f:
    ckpt = pickle.load(f)

params = ckpt['params']
W_rec  = np.array(params['W_rec'])   # (n_rnn, n_rnn)  W_rec[i,j] = i → j
W_in   = np.array(params['W_in'])    # (n_in,  n_rnn)
W_out  = np.array(params['W_out'])   # (n_rnn, n_out)
b_rec  = np.array(params['b_rec'])   # (n_rnn,)
n_rnn  = W_rec.shape[0]
n_in   = W_in.shape[0]

print(f'Checkpoint : {CKPT.name}  (step {ckpt.get("step", "?")})')
print(f'W_rec      : {W_rec.shape}   W_in: {W_in.shape}   W_out: {W_out.shape}')
print(f'W_rec stats: min={W_rec.min():.3f}  max={W_rec.max():.3f}  std={W_rec.std():.3f}')

## 1. Excitatory / Inhibitory classification
Row sum of W_rec = net weight a neuron sends out. Positive → net excitatory output.

In [ ]:
row_sum = W_rec.sum(axis=1)   # (n_rnn,)  net outgoing weight
col_sum = W_rec.sum(axis=0)   # (n_rnn,)  net incoming weight
is_exc  = row_sum > 0

print(f'Excitatory (row_sum > 0): {is_exc.sum()}/{n_rnn}')
print(f'Inhibitory (row_sum ≤ 0): {(~is_exc).sum()}/{n_rnn}')

ei_color = np.where(is_exc, 'tab:red', 'tab:blue')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.hist(row_sum[is_exc],  bins=25, color='tab:red',  alpha=0.7, label='Exc (row>0)', edgecolor='white')
ax.hist(row_sum[~is_exc], bins=25, color='tab:blue', alpha=0.7, label='Inh (row≤0)', edgecolor='white')
ax.axvline(0, color='k', lw=1, ls='--')
ax.set(xlabel='Row sum of W_rec', ylabel='Neuron count', title='Net outgoing weight distribution')
ax.legend()

ax = axes[1]
ax.scatter(row_sum, col_sum, c=ei_color, s=20, alpha=0.7, edgecolors='none')
ax.axhline(0, color='k', lw=0.7, ls='--')
ax.axvline(0, color='k', lw=0.7, ls='--')
ax.set(xlabel='Row sum (net outgoing)', ylabel='Col sum (net incoming)',
       title='Outgoing vs Incoming net weight')
ax.legend(handles=[mpatches.Patch(color='tab:red',  label='Exc (row>0)'),
                   mpatches.Patch(color='tab:blue', label='Inh (row≤0)')], fontsize=9)

ax = axes[2]
ax.scatter(row_sum, W_out[:, 0], c=ei_color, s=20, alpha=0.7, edgecolors='none')
ax.axhline(0, color='k', lw=0.7, ls='--')
ax.axvline(0, color='k', lw=0.7, ls='--')
ax.set(xlabel='Row sum (net outgoing)', ylabel='W_out weight (→ readout)',
       title='E/I vs Readout contribution')

plt.tight_layout()
plt.show()

## 2. Hierarchical clustering by connectivity profile
Neurons are clustered by their **outgoing weight profile** (rows of W_rec) using cosine distance
so cluster membership reflects *who* each neuron targets, not the magnitude of its weights.

In [ ]:
dist_vec = pdist(W_rec, metric='cosine')
Z_link   = linkage(dist_vec, method=LINK_METHOD)

fig, ax = plt.subplots(figsize=(14, 3.5))
dendrogram(Z_link, ax=ax, no_labels=True, color_threshold=0,
           above_threshold_color='gray',
           link_color_func=lambda k: 'steelblue')
# show cut height for N_CLUSTERS
cut_h = (Z_link[-N_CLUSTERS + 1, 2] + Z_link[-N_CLUSTERS, 2]) / 2
ax.axhline(cut_h, color='tomato', ls='--', lw=1.2,
           label=f'Cut → N={N_CLUSTERS} clusters')
ax.set(xlabel='Neurons', ylabel='Cosine distance (Ward linkage)',
       title='Dendrogram — outgoing connectivity profile')
ax.legend()
plt.tight_layout()
plt.show()

# Assign clusters (scipy returns 1-indexed → shift to 0)
cluster_labels = fcluster(Z_link, N_CLUSTERS, criterion='maxclust') - 1

# Re-order cluster IDs: most inhibitory (lowest mean row_sum) → 0
cl_mean_rs = np.array([row_sum[cluster_labels == c].mean() for c in range(N_CLUSTERS)])
order      = np.argsort(cl_mean_rs)
remap      = {old: new for new, old in enumerate(order)}
cluster_labels = np.array([remap[c] for c in cluster_labels])
cluster_colors = plt.cm.tab10(np.linspace(0, 0.9, N_CLUSTERS))

print('Cluster sizes:', {c: int((cluster_labels == c).sum()) for c in range(N_CLUSTERS)})
print('Mean row_sum per cluster:', {c: f'{row_sum[cluster_labels==c].mean():.3f}' for c in range(N_CLUSTERS)})

## 3. Sorted W_rec heatmap
Rows and columns re-ordered by cluster, then by row_sum within cluster.
If clusters are real, you expect dense blocks on the diagonal.

In [ ]:
sort_idx     = np.lexsort((row_sum, cluster_labels))
W_sorted     = W_rec[np.ix_(sort_idx, sort_idx)]
labels_sorted = cluster_labels[sort_idx]

boundaries = np.where(np.diff(labels_sorted))[0] + 1
starts     = np.concatenate([[0], boundaries, [n_rnn]])

vlim = np.percentile(np.abs(W_rec), 98)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(W_sorted, cmap='RdBu_r', vmin=-vlim, vmax=vlim,
               aspect='auto', interpolation='nearest')
for b in boundaries:
    ax.axhline(b - 0.5, color='k', lw=0.8)
    ax.axvline(b - 0.5, color='k', lw=0.8)
for c in range(N_CLUSTERS):
    mid = (starts[c] + starts[c + 1]) / 2
    ax.text(mid, mid, f'C{c}', ha='center', va='center',
            fontsize=12, fontweight='bold', color='white',
            bbox=dict(boxstyle='round,pad=0.2', fc=cluster_colors[c], alpha=0.9, ec='none'))
plt.colorbar(im, ax=ax, label='Weight', shrink=0.7, pad=0.02)
ax.set(xlabel='Target neuron (sorted by cluster)',
       ylabel='Source neuron (sorted by cluster)',
       title='W_rec sorted by connectivity cluster')
plt.tight_layout()
plt.show()

## 4. Within- vs Between-cluster connectivity
Tests whether clusters have higher internal connectivity — the brain-region criterion.

In [ ]:
within_mean, between_mean = [], []
for c in range(N_CLUSTERS):
    m = cluster_labels == c
    within_mean.append( np.abs(W_rec[np.ix_( m,  m)]).mean())
    between_mean.append(np.abs(W_rec[np.ix_( m, ~m)]).mean())

within_mean  = np.array(within_mean)
between_mean = np.array(between_mean)

# Global: fraction of total |weight| that falls within clusters
same_cl = np.equal.outer(cluster_labels, cluster_labels)
np.fill_diagonal(same_cl, False)
off_diag = ~np.eye(n_rnn, dtype=bool)
frac_within = np.abs(W_rec[same_cl]).sum() / np.abs(W_rec[off_diag]).sum()
# Expected fraction if clusters were random (proportional to cluster sizes)
expected = sum((cluster_labels == c).sum() ** 2 for c in range(N_CLUSTERS))
expected_frac = (expected - n_rnn) / (n_rnn ** 2 - n_rnn)

print(f'Fraction of |weight| within clusters : {frac_within:.3f}')
print(f'Expected under random assignment      : {expected_frac:.3f}')
print(f'Modularity ratio (observed/expected)  : {frac_within / expected_frac:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
x, w = np.arange(N_CLUSTERS), 0.35
ax.bar(x - w/2, within_mean,  w, color='steelblue', label='Within-cluster')
ax.bar(x + w/2, between_mean, w, color='tomato',    label='Between-cluster')
ax.set(xlabel='Cluster', ylabel='Mean |weight|',
       title='Within vs Between connectivity',
       xticks=x, xticklabels=[f'C{c}' for c in range(N_CLUSTERS)])
ax.legend()

ax = axes[1]
ratio = within_mean / (between_mean + 1e-12)
bars  = ax.bar(x, ratio, color=cluster_colors)
ax.axhline(1.0, color='k', lw=0.8, ls='--', label='within = between')
for bar, r in zip(bars, ratio):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{r:.2f}', ha='center', va='bottom', fontsize=9)
ax.set(xlabel='Cluster', ylabel='Within / Between ratio',
       title='Modularity ratio by cluster',
       xticks=x, xticklabels=[f'C{c}' for c in range(N_CLUSTERS)])
ax.legend()

plt.tight_layout()
plt.show()

## 5. Cluster functional profiles
For each cluster: mean W_in channel weights (input selectivity) and W_out distribution (readout contribution).

In [ ]:
# W_in columns are neurons; transpose so rows are neurons
W_in_per_neuron = W_in.T   # (n_rnn, n_in)
n_freq = task_params.get('n_freq', n_in - 1)
ch_labels = [f'f{i+1}' for i in range(n_freq)] + ['fix']

fig, axes = plt.subplots(2, N_CLUSTERS, figsize=(4 * N_CLUSTERS, 7), sharey='row')

for c in range(N_CLUSTERS):
    mask = cluster_labels == c
    n_c  = mask.sum()
    col  = cluster_colors[c]

    # --- W_in profile (top row) ---
    ax = axes[0, c]
    win_mean = W_in_per_neuron[mask].mean(axis=0)
    win_sem  = W_in_per_neuron[mask].std(axis=0, ddof=1) / np.sqrt(n_c)
    ax.bar(np.arange(n_in), win_mean, color=col, alpha=0.8)
    ax.errorbar(np.arange(n_in), win_mean, yerr=win_sem, fmt='none',
                color='k', capsize=3, elinewidth=0.8)
    ax.axhline(0, color='k', lw=0.6)
    ax.set_xticks(np.arange(n_in))
    ax.set_xticklabels(ch_labels[:n_in], rotation=45, fontsize=8)
    ax.set_title(f'C{c}  (n={n_c})', color=col, fontweight='bold')
    if c == 0:
        ax.set_ylabel('Mean W_in  (mean ± SEM across neurons)')

    # --- W_out distribution (bottom row) ---
    ax = axes[1, c]
    wout = W_out[mask, 0]
    ax.hist(wout, bins=15, color=col, alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.axvline(0,           color='k', lw=0.8, ls='--')
    ax.axvline(wout.mean(), color='k', lw=1.2, ls='-',
               label=f'mean = {wout.mean():.3f}')
    ax.legend(fontsize=8)
    ax.set_xlabel('W_out weight (→ readout)')
    if c == 0:
        ax.set_ylabel('Neuron count')

fig.suptitle('Cluster profiles: input weights (top) and readout weights (bottom)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6. PCA of connectivity space
Embed each neuron by its outgoing weight vector; colour by cluster (left) and E/I row-sum (right).

In [ ]:
pca  = PCA(n_components=3)
emb  = pca.fit_transform(W_rec)   # (n_rnn, 3)
ve   = pca.explained_variance_ratio_ * 100
print(f'PCA variance explained (connectivity): PC1={ve[0]:.1f}%  PC2={ve[1]:.1f}%  PC3={ve[2]:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for c in range(N_CLUSTERS):
    m = cluster_labels == c
    ax.scatter(emb[m, 0], emb[m, 1], s=30, alpha=0.8,
               color=cluster_colors[c], label=f'C{c}', edgecolors='none')
ax.set(xlabel=f'PC1 ({ve[0]:.1f}%)', ylabel=f'PC2 ({ve[1]:.1f}%)',
       title='Connectivity PCA — by cluster')
ax.legend(markerscale=1.5)

ax = axes[1]
vmax = np.percentile(np.abs(row_sum), 95)
sc   = ax.scatter(emb[:, 0], emb[:, 1], c=row_sum,
                  cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                  s=30, alpha=0.8, edgecolors='none')
plt.colorbar(sc, ax=ax, label='Row sum (net outgoing weight)')
ax.set(xlabel=f'PC1 ({ve[0]:.1f}%)', ylabel=f'PC2 ({ve[1]:.1f}%)',
       title='Connectivity PCA — E/I (row sum)')

plt.tight_layout()
plt.show()

## 7. Cross-cluster connectivity matrix
Aggregate view: mean weight from each cluster to each other cluster.

In [ ]:
# Build C × C matrix of mean weights (signed) and mean absolute weights
CC_mean = np.zeros((N_CLUSTERS, N_CLUSTERS))
CC_abs  = np.zeros((N_CLUSTERS, N_CLUSTERS))
for ci in range(N_CLUSTERS):
    for cj in range(N_CLUSTERS):
        block = W_rec[np.ix_(cluster_labels == ci, cluster_labels == cj)]
        CC_mean[ci, cj] = block.mean()
        CC_abs[ci, cj]  = np.abs(block).mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, mat, title, cmap in [
    (axes[0], CC_mean, 'Mean signed weight  (source → target)', 'RdBu_r'),
    (axes[1], CC_abs,  'Mean |weight|  (connection strength)',  'viridis'),
]:
    vlim = np.abs(mat).max()
    kwargs = dict(vmin=-vlim, vmax=vlim) if cmap == 'RdBu_r' else dict(vmin=0, vmax=vlim)
    im = ax.imshow(mat, cmap=cmap, **kwargs)
    plt.colorbar(im, ax=ax, shrink=0.85)
    ax.set_xticks(range(N_CLUSTERS)); ax.set_yticks(range(N_CLUSTERS))
    ax.set_xticklabels([f'C{c}' for c in range(N_CLUSTERS)])
    ax.set_yticklabels([f'C{c}' for c in range(N_CLUSTERS)])
    for i in range(N_CLUSTERS):
        for j in range(N_CLUSTERS):
            ax.text(j, i, f'{mat[i, j]:.3f}', ha='center', va='center',
                    fontsize=9, color='white' if abs(mat[i,j]) > 0.5 * vlim else 'black')
    ax.set(xlabel='Target cluster', ylabel='Source cluster', title=title)

plt.tight_layout()
plt.show()

## 8. Neuron activity during the task

Run the model on task trials, align every trial to its go cue, then compare
activity of each **connectivity cluster** across Go vs No-Go conditions.

- Thin coloured lines = individual neurons (trial-averaged)
- Thick line + shading = cluster mean ± SEM across neurons

In [ ]:
sys.path.insert(0, str(Path.cwd()))
from network import LeakyRNN
from task import generate_trials
import jax
import jax.numpy as jnp

TASK_NAME = 'harmonic'   # change to 'pure', 'wn', etc.
N_TRIALS  = 256
SEED_ACT  = 0
PRE_MS    = 200   # ms before go cue
POST_MS   = 400   # ms after  go cue

dt        = float(ckpt.get('dt', task_params['timing']['dt']))
alpha_rnn = dt / hp['model']['tau']
pre_steps  = int(PRE_MS  / dt)
post_steps = int(POST_MS / dt)

model = LeakyRNN(n_rnn=n_rnn, n_out=1, alpha=alpha_rnn,
                 sigma_rec=hp['model']['sigma_rec'],
                 sigma_x=hp['model']['sigma_x'])

task_list = config.get('task_list', config.get('tasks', ['pure', 'harmonic', 'wn']))
trial = generate_trials(TASK_NAME, task_list, mode='test',
                        batch_size=N_TRIALS,
                        rng=np.random.default_rng(SEED_ACT))

_, hs_jax = model.apply({'params': params}, jnp.array(trial.x),
                         noise_input=False, noise_rec=False,
                         rngs={'noise': jax.random.PRNGKey(0)})
hs    = np.array(hs_jax)   # (T, batch, n_rnn)
is_go = trial.is_go         # (batch,) bool

# stim_on may be per-trial array or scalar depending on mode
stim_on_arr = np.asarray(trial.stim_on).ravel()
if stim_on_arr.size == 1:
    stim_on_arr = np.full(N_TRIALS, int(stim_on_arr[0]))

valid = (stim_on_arr >= pre_steps) & (stim_on_arr + post_steps <= hs.shape[0])
hs_aln = np.stack([
    hs[so - pre_steps : so + post_steps, i, :]
    for i, so in enumerate(stim_on_arr) if valid[i]
], axis=1)           # (T_aln, n_valid, n_rnn)
is_go_v = is_go[valid]
t_ms    = np.arange(-pre_steps, post_steps) * dt   # ms from go cue

print(f'Task: {TASK_NAME}   aligned shape: {hs_aln.shape}')
print(f'Go: {is_go_v.sum()}   No-Go: {(~is_go_v).sum()}')

In [ ]:
# ── Mean activity per cluster: Go (left) vs No-Go (right) ────────────────────
fig, axes = plt.subplots(N_CLUSTERS, 2, figsize=(13, 3.8 * N_CLUSTERS),
                          sharex=True, sharey='row')

for c in range(N_CLUSTERS):
    cl_idx = np.where(cluster_labels == c)[0]
    hs_cl  = hs_aln[:, :, cl_idx]               # (T, n_valid, n_cl)

    for col_i, (mask_cond, color, lbl) in enumerate([
        (is_go_v,  'tab:blue',   'Go'),
        (~is_go_v, 'tab:orange', 'No-Go'),
    ]):
        ax      = axes[c, col_i]
        hs_cond = hs_cl[:, mask_cond, :]          # (T, n_cond, n_cl)
        tr_mean = hs_cond.mean(axis=1)            # (T, n_cl) — averaged over trials

        ax.plot(t_ms, tr_mean, color=color, alpha=0.12, lw=0.6)  # individual neurons
        m = tr_mean.mean(axis=1)
        s = tr_mean.std(axis=1, ddof=1) / np.sqrt(max(len(cl_idx) - 1, 1))
        ax.plot(t_ms, m, color=color, lw=2)
        ax.fill_between(t_ms, m - s, m + s, color=color, alpha=0.3)
        ax.axvline(0, color='k', lw=0.7, ls='--')
        ax.set_title(f'C{c}  ({len(cl_idx)} neurons) — {lbl}',
                     color=cluster_colors[c], fontweight='bold')
        if col_i == 0:
            ax.set_ylabel('Hidden state')
        if c == N_CLUSTERS - 1:
            ax.set_xlabel('Time from go cue (ms)')

plt.suptitle(f'Activity by connectivity cluster  |  task = {TASK_NAME}',
             fontsize=13, y=1.005)
plt.tight_layout()
plt.show()

In [ ]:
# ── Go − No-Go difference per cluster ────────────────────────────────────────
fig, axes = plt.subplots(1, N_CLUSTERS, figsize=(4.5 * N_CLUSTERS, 4),
                          sharex=True, sharey=True)

for c, ax in enumerate(axes):
    cl_idx    = np.where(cluster_labels == c)[0]
    hs_cl     = hs_aln[:, :, cl_idx]
    go_mean   = hs_cl[:, is_go_v,  :].mean(axis=1)   # (T, n_cl)
    nogo_mean = hs_cl[:, ~is_go_v, :].mean(axis=1)
    diff      = go_mean - nogo_mean
    m = diff.mean(axis=1)
    s = diff.std(axis=1, ddof=1) / np.sqrt(max(len(cl_idx) - 1, 1))
    ax.plot(t_ms, m, color=cluster_colors[c], lw=2)
    ax.fill_between(t_ms, m - s, m + s, color=cluster_colors[c], alpha=0.3)
    ax.axhline(0, color='k', lw=0.7, ls='--')
    ax.axvline(0, color='k', lw=0.7, ls='--')
    ax.set_title(f'C{c}  ({len(cl_idx)} neurons)',
                 color=cluster_colors[c], fontweight='bold')
    ax.set_xlabel('Time from go cue (ms)')
    if c == 0:
        ax.set_ylabel('Δ activity  (Go − No-Go)')

plt.suptitle(f'Go − No-Go activity difference by cluster  |  {TASK_NAME}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Individual-neuron heatmap, sorted by cluster then peak time ───────────────
# z-score each neuron across time using the all-trial mean
hs_all_mean = hs_aln.mean(axis=1)                           # (T_aln, n_rnn)
mu_n  = hs_all_mean.mean(axis=0, keepdims=True)
sig_n = hs_all_mean.std(axis=0,  keepdims=True) + 1e-8

stim_win = t_ms > 0
sort_idx_act = []
for c in range(N_CLUSTERS):
    cl_idx = np.where(cluster_labels == c)[0]
    peak_t = hs_all_mean[stim_win][:, cl_idx].argmax(axis=0)
    sort_idx_act.extend(cl_idx[np.argsort(peak_t)])
sort_idx_act = np.array(sort_idx_act)

labels_act = cluster_labels[sort_idx_act]
bounds_act = np.where(np.diff(labels_act))[0] + 1
starts_act = np.concatenate([[0], bounds_act, [n_rnn]])

fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)

for ax, mask_cond, title in [
    (axes[0], is_go_v,  f'Go  (n={is_go_v.sum()})'),
    (axes[1], ~is_go_v, f'No-Go  (n={(~is_go_v).sum()})'),
]:
    # z-score using global mu/sig so the two panels are comparable
    raw = hs_aln[:, mask_cond, :][:, :, sort_idx_act].mean(axis=1)  # (T, n_rnn)
    z   = (raw - mu_n[:, sort_idx_act]) / sig_n[:, sort_idx_act]     # (T, n_rnn)

    im = ax.imshow(z.T, aspect='auto', cmap='RdBu_r', vmin=-2.5, vmax=2.5,
                   extent=[t_ms[0], t_ms[-1], n_rnn, 0])
    for b in bounds_act:
        ax.axhline(b - 0.5, color='k', lw=0.8)
    for c in range(N_CLUSTERS):
        mid = (starts_act[c] + starts_act[c + 1]) / 2
        ax.text(t_ms[-1] + 5, mid, f'C{c}', ha='left', va='center',
                color=cluster_colors[c], fontweight='bold', fontsize=10,
                transform=ax.get_yaxis_transform())
    ax.axvline(0, color='yellow', lw=0.9, ls='--', alpha=0.8)
    plt.colorbar(im, ax=ax, label='z-scored activity', shrink=0.7)
    ax.set(xlabel='Time from go cue (ms)',
           ylabel='Neuron  (sorted: cluster → peak time)',
           title=title)

plt.suptitle(f'Single-neuron activity heatmap by connectivity cluster  |  {TASK_NAME}',
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── E/I breakdown within each cluster ────────────────────────────────────────
# solid = Excitatory neurons, dashed = Inhibitory; blue = Go, orange = No-Go
fig, axes = plt.subplots(1, N_CLUSTERS, figsize=(4.5 * N_CLUSTERS, 4),
                          sharex=True, sharey=True)

for c, ax in enumerate(axes):
    cl_idx = np.where(cluster_labels == c)[0]

    for ei_mask_local, ei_lbl, lstyle in [
        (is_exc[cl_idx],  'Exc', '-'),
        (~is_exc[cl_idx], 'Inh', '--'),
    ]:
        if ei_mask_local.sum() == 0:
            continue
        sub_idx = cl_idx[ei_mask_local]

        for mask_cond, color, cond_lbl in [
            (is_go_v,  'tab:blue',   'Go'),
            (~is_go_v, 'tab:orange', 'No-Go'),
        ]:
            m = hs_aln[:, mask_cond, :][:, :, sub_idx].mean(axis=(1, 2))  # (T,)
            ax.plot(t_ms, m, color=color, ls=lstyle, lw=1.8,
                    label=f'{ei_lbl} {cond_lbl} (n={ei_mask_local.sum()})')

    ax.axvline(0, color='k', lw=0.7, ls=':')
    ax.axhline(0, color='k', lw=0.4)
    ax.set_title(f'C{c}  ({len(cl_idx)} neurons)',
                 color=cluster_colors[c], fontweight='bold')
    ax.set_xlabel('Time from go cue (ms)')
    if c == 0:
        ax.set_ylabel('Mean hidden state')
    ax.legend(fontsize=7, loc='upper left')

plt.suptitle(f'E/I activity within clusters  |  {TASK_NAME}'
             '  |  solid = Exc (row_sum > 0)   dashed = Inh',
             fontsize=11)
plt.tight_layout()
plt.show()